In [ ]:
#CORRETO
import os
import librosa
import numpy as np
from sklearn.svm import SVC  # Trocamos para SVM Linear
from pydub import AudioSegment
from pathlib import Path

def garantir_formato_wav(caminho_arquivo):
    """
    Verifica se o arquivo já é .wav. Se não for, converte usando o pydub
    e retorna o caminho para o novo arquivo .wav.
    """
    caminho = Path(caminho_arquivo)

    # Se já for .wav, apenas retorna o caminho original
    if caminho.suffix.lower() == '.wav':
        return caminho

    # Define o novo caminho com a extensão .wav
    caminho_wav = caminho.with_suffix('.wav')

    print(f"Convertendo '{caminho.name}' para formato WAV...")
    try:
        # Extrai o formato original (ex: 'mp3', 'm4a') tirando o ponto inicial
        formato_original = caminho.suffix.lower()[1:]

        # Carrega e exporta para wav
        audio = AudioSegment.from_file(caminho, format=formato_original)
        audio.export(caminho_wav, format="wav")

        return caminho_wav
    except Exception as e:
        print(f"Erro ao converter '{caminho.name}': {e}")
        return None

def _compute_features_from_audio(y, sr):
    """
    Função interna que extrai features temporais (média, desvio, inclinação)
    das MFCCs, Deltas, Delta-Deltas e Centróide, divididos em 3 segmentos.
    CORRIGIDA: agora calcula o slope feature por feature para evitar erro de dimensão.
    """
    # Extrai as bases espectrais
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    delta1 = librosa.feature.delta(mfccs)
    delta2 = librosa.feature.delta(mfccs, order=2)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)

    # Empilha tudo: (13 MFCC + 13 Delta + 13 Delta2 + 1 Centroid) = 40 features
    feat_stack = np.vstack([mfccs, delta1, delta2, centroid])
    T = feat_stack.shape[1]  # Número de frames temporais

    # Divide o áudio em 3 partes (início, meio, fim)
    if T < 3:
        segments = [np.arange(T)]
    else:
        segments = np.array_split(np.arange(T), 3)

    all_stats = []
    for idxs in segments:
        if len(idxs) == 0:
            continue
        seg_feat = feat_stack[:, idxs]  # shape: (n_feats, n_frames_no_segmento)

        # Média (sobre o eixo temporal)
        mean_vals = np.mean(seg_feat, axis=1)
        # Desvio padrão
        std_vals = np.std(seg_feat, axis=1)

        # Inclinação (slope) - CALCULADA CORRETAMENTE LINHA POR LINHA
        n_frames = seg_feat.shape[1]
        if n_frames > 1:
            x_vals = np.arange(n_frames)
            # Para cada feature (linha), ajusta uma reta e pega o coeficiente angular
            slopes = np.array([
                np.polyfit(x_vals, seg_feat[i, :], 1)[0]
                for i in range(seg_feat.shape[0])
            ])
        else:
            slopes = np.zeros(seg_feat.shape[0])

        all_stats.extend(mean_vals)
        all_stats.extend(std_vals)
        all_stats.extend(slopes)

    return np.array(all_stats)

def extrair_caracteristicas(caminho_pro_arquivo):
    """
    Mantém a mesma assinatura original: recebe um caminho e retorna um vetor numpy.
    Agora com taxa fixa, trim, normalização e features temporais.
    """
    # Taxa fixa (22050 Hz) para consistência entre todos os áudios
    y, sr = librosa.load(caminho_pro_arquivo, sr=22050)

    # Remove silêncio nas bordas (evita que o modelo aprenda com silêncio)
    y, _ = librosa.effects.trim(y, top_db=20)

    # Normaliza a amplitude para evitar diferenças de volume
    if np.max(np.abs(y)) > 0:
        y = y / np.max(np.abs(y))

    return _compute_features_from_audio(y, sr)


# 1. Configuração das categorias (pastas de treinamento)
categorias = ['feliz', 'questionamento', 'triste', 'brava']

X_treino = []
y_treino = []

# Extensões de áudio originais que o script vai procurar
extensoes_permitidas = ['.wav', '.mp3', '.m4a', '.ogg', '.flac']

print("Buscando e processando arquivos de treinamento por categoria...")
print("(Aplicando aumento de dados: pitch shift e time stretch)")

# 2. Percorre cada pasta de categoria
for categoria in categorias:
    pasta_categoria = Path(categoria)

    # Verifica se a pasta daquela categoria existe
    if not pasta_categoria.exists() or not pasta_categoria.is_dir():
        print(f"Aviso: A pasta '{categoria}' não foi encontrada. Pulando esta categoria.")
        continue

    print(f"\n--- Lendo arquivos da categoria: {categoria.upper()} ---")

    for caminho_arquivo in pasta_categoria.iterdir():
        if caminho_arquivo.is_file() and caminho_arquivo.suffix.lower() in extensoes_permitidas:

            # Passa pela função para garantir que seja WAV
            caminho_wav = garantir_formato_wav(caminho_arquivo)

            if caminho_wav is None:
                continue # Pula este arquivo se houve erro na conversão

            print(f"Extraindo características de: {caminho_wav.name}")

            # --- Carrega o áudio UMA VEZ para fazer aumento de dados ---
            try:
                y, sr = librosa.load(caminho_wav, sr=22050)
                y, _ = librosa.effects.trim(y, top_db=20)
                if np.max(np.abs(y)) > 0:
                    y = y / np.max(np.abs(y))
            except Exception as e:
                print(f"Erro ao carregar {caminho_wav.name}: {e}")
                continue

            # 1. Áudio original
            feat_original = _compute_features_from_audio(y, sr)
            X_treino.append(feat_original)
            y_treino.append(categoria)

            # 2. Aumento de dados: Pitch shift (+2 e -2 semitons)
            try:
                y_pitch_up = librosa.effects.pitch_shift(y, sr=sr, n_steps=2)
                X_treino.append(_compute_features_from_audio(y_pitch_up, sr))
                y_treino.append(categoria)
            except:
                pass
            try:
                y_pitch_down = librosa.effects.pitch_shift(y, sr=sr, n_steps=-2)
                X_treino.append(_compute_features_from_audio(y_pitch_down, sr))
                y_treino.append(categoria)
            except:
                pass

            # 3. Aumento de dados: Time stretch (0.9x e 1.1x)
            try:
                y_stretch_slow = librosa.effects.time_stretch(y, rate=0.9)
                X_treino.append(_compute_features_from_audio(y_stretch_slow, sr))
                y_treino.append(categoria)
            except:
                pass
            try:
                y_stretch_fast = librosa.effects.time_stretch(y, rate=1.1)
                X_treino.append(_compute_features_from_audio(y_stretch_fast, sr))
                y_treino.append(categoria)
            except:
                pass

# Verifica se encontrou algum arquivo para treinar no geral
if len(X_treino) == 0:
    print("\nNenhum arquivo de áudio válido encontrado nas pastas de categorias.")
else:
    print(f"\nTotal de amostras geradas (com aumento): {len(X_treino)}")
    print("Treinando o modelo com SVM Linear...")

    # 3. Inicializa e treina o classificador (SVM Linear é mais robusto com poucos dados)
    # Mantive o nome 'classificador' exatamente como antes
    classificador = SVC(kernel='linear', class_weight='balanced', random_state=42)
    classificador.fit(X_treino, y_treino)
    print("Treinamento concluído!\n")

Buscando e processando arquivos de treinamento por categoria...
(Aplicando aumento de dados: pitch shift e time stretch)

--- Lendo arquivos da categoria: FELIZ ---
Extraindo características de: Cidade Universitária Armando Salles de Oliveira 354.wav
Extraindo características de: Cidade Universitária Armando Salles de Oliveira 359.wav
Extraindo características de: Cidade Universitária Armando Salles de Oliveira 356.wav
Extraindo características de: Cidade Universitária Armando Salles de Oliveira 353.wav
Extraindo características de: Cidade Universitária Armando Salles de Oliveira 360.wav
Convertendo 'Cidade Universitária Armando Salles de Oliveira 358.m4a' para formato WAV...
Extraindo características de: Cidade Universitária Armando Salles de Oliveira 358.wav
Convertendo 'Cidade Universitária Armando Salles de Oliveira 359.m4a' para formato WAV...
Extraindo características de: Cidade Universitária Armando Salles de Oliveira 359.wav
Convertendo 'Cidade Universitária Armando S

In [ ]:
# 4. --- Predição de um novo áudio ---
# Coloque o caminho do arquivo que você quer testar aqui

garantir_formato_wav("triste_teste.ogg")
arquivo_teste = "triste_teste.wav"

if os.path.exists(arquivo_teste):
    print(f"Analisando arquivo de teste: {arquivo_teste}")
    # Reaproveitamos a função para o arquivo de teste
    caminho_teste_wav = garantir_formato_wav(arquivo_teste)

    if caminho_teste_wav:
        # Extrai características e prevê
        novo_som = extrair_caracteristicas(caminho_teste_wav)
        humor_detectado = classificador.predict([novo_som])

        print(f"\n>>> O estado emocional estimado do animal é: {humor_detectado[0].upper()} <<<")
else:
    print(f"Erro: O arquivo de teste '{arquivo_teste}' não foi encontrado.")

Convertendo 'triste_teste.ogg' para formato WAV...
Analisando arquivo de teste: triste_teste.wav

>>> O estado emocional estimado do animal é: TRISTE <<<
